# IMTalker on RunPod — image + audio → talking-head video (inference only)

One reference photo and one audio file in, one lip-synced MP4 out. This notebook runs
**only the IMTalker avatar model**: the flow-matching motion generator, its renderer and
the frozen Wav2Vec2 audio encoder. None of the speech-to-speech stack (PersonaPlex 7B,
Moshi, Mimi, the TensorRT SEANet decoder, the websocket server) is downloaded or loaded —
that is what `runpod_speech2face.ipynb` is for.

Defaults, both already committed in the repo:

| | |
|---|---|
| image | `src/imtalker/image/rb.jpeg` |
| audio | `src/imtalker/audio/rb_wav.wav` |

### Before you start

| | |
|---|---|
| **GPU** | any NVIDIA GPU with compute capability ≥ 7.0 and ~8 GB of VRAM. No int4 kernels and no TensorRT here, so Turing works too |
| **Disk** | **≥ 20 GB** on `/workspace` (Python env ~5 GB, IMTalker weights ~3 GB, HF cache) |
| **Pod template** | any RunPod *PyTorch* or *CUDA 12* image — the notebook builds its own Python 3.10 env, so the base image's Python does not matter |
| **HF token** | **not needed.** `cbsjtu01/IMTalker` is public and ungated |
| **Exposed ports** | none. Nothing is served; the video is written to a file and played inline |

### What the cells do

1. configuration (paths, sampling knobs) — the only cell you normally edit
2. check the GPU, driver and disk
3. install the handful of system libraries (`ffmpeg`, `libsndfile1`, `libgl1`)
4. clone the repo into `/workspace/speech2face_imtalker`
5. write the inference-only requirement files + `setup_env_imtalker.sh`
6. build the Python 3.10 env in `/workspace/imtalker_only/env` (~5–12 min)
7. download `generator.ckpt`, `renderer.ckpt` and `wav2vec2-base-960h` (~3 GB, public)
8. write `imtalker_infer.py`
9. check the inputs and preview the reference frame
10. generate the video
11. play it inline and get a download path
12. knobs, batching and troubleshooting

First run: roughly 15–25 minutes, nearly all of it downloading. Everything lands on
`/workspace`, so a restarted pod with the same volume skips steps 6 and 7 and reaches a
video in under a minute.


## 1 · Configuration

The paths are the only thing you normally change. `IMAGE` and `AUDIO` may be anything
readable — a path inside the clone, something you uploaded to `/workspace`, or an absolute
path on a mounted volume.

`NFE` / `A_CFG_SCALE` / `SEED` are the sampler knobs; section 12 explains what moving them
does. `CROP = False` is right for a photo that is already a square head-and-shoulders
portrait, which `rb.jpeg` (512×512) is. Set it to `True` for a wider snapshot.


In [ ]:
import os, subprocess, sys, time, json, shutil, textwrap
from pathlib import Path

# ----------------------------------------------------------------- settings
REPO_URL = "https://github.com/MoshiHead/speech2face_orin_n_original_IMTalker_inference_only.git"
REPO_DIR = "/workspace/speech2face_imtalker"   # the clone lands here
BRANCH   = ""                                  # "" = the repo's default branch
WORK     = "/workspace/imtalker_only"          # env, checkpoints and outputs live here

# inputs -- relative paths are resolved against the clone
IMAGE = "src/imtalker/image/rb.jpeg"
AUDIO = "src/imtalker/audio/rb_wav.wav"
OUT   = "outputs/rb_imtalker.mp4"              # relative paths are resolved against WORK

# sampler
NFE          = 10     # ODE steps of the flow-matching sampler (5 fast .. 25 smooth)
A_CFG_SCALE  = 2.0    # audio classifier-free guidance (1.0 flat .. 3.0 exaggerated)
SEED         = 42
CROP         = False  # True = auto-crop around the detected face first (needs face_alignment)
FPS          = 25.0   # the generator is trained at 25 fps -- changing this desyncs the lips
CRF          = 18     # x264 quality of the muxed output (lower = bigger + better)

# checkpoints: the public, ungated IMTalker release
HF_CKPT_REPO = "cbsjtu01/IMTalker"
# Point these at your own fine-tuned files to use them instead; "" = use the HF download.
GENERATOR_OVERRIDE = ""
RENDERER_OVERRIDE  = ""

# ----------------------------------------------------------------- derived paths
WORK = os.path.abspath(WORK)
CKPT_DIR = os.path.join(WORK, "checkpoints")
PY_ENV   = os.path.join(WORK, "env", "bin", "python")
IMTALKER_DIR = os.path.join(REPO_DIR, "src", "imtalker")
os.makedirs(WORK, exist_ok=True)
os.makedirs(os.path.join(WORK, "outputs"), exist_ok=True)

# keep the HF cache on the persistent volume, not the container disk
os.environ.setdefault("HF_HOME", os.path.join(WORK, "hf"))
# huggingface_hub refuses to download at all when RunPod's images export
# HF_HUB_ENABLE_HF_TRANSFER=1 and the hf_transfer package is not installed
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

def resolve(p, base):
    """Absolute paths pass through; relative ones are resolved against `base`."""
    return p if os.path.isabs(p) else os.path.join(base, p)

# ----------------------------------------------------------------- helper
def sh(cmd, cwd=None, check=True, env=None, quiet=False):
    """Run a shell command, streaming its output into the notebook as it happens."""
    cwd = cwd or (WORK if os.path.isdir(WORK) else os.getcwd())
    if not quiet:
        print("$ " + cmd, flush=True)
    p = subprocess.Popen(["bash", "-c", cmd], cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1,
                         errors="replace", env={**os.environ, **(env or {})})
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
    if check and rc != 0:
        raise RuntimeError("command failed (exit %d): %s" % (rc, cmd))
    return rc

# every later %%writefile cell is relative to WORK
os.chdir(WORK)

print("repo        :", REPO_URL)
print("clone into  :", REPO_DIR)
print("work dir    :", WORK)
print("checkpoints :", CKPT_DIR, "(from %s)" % HF_CKPT_REPO)
print("env python  :", PY_ENV)
print("sampler     : nfe=%d a_cfg_scale=%.2f seed=%d crop=%s fps=%g"
      % (NFE, A_CFG_SCALE, SEED, CROP, FPS))


## 2 · Check the GPU, driver and disk

A driver older than 525 cannot run the CUDA 12.8 wheels, and less than ~20 GB free on
`/workspace` will stop you at step 6 or 7 — better to see it now. Unlike the full
speech-to-speech notebook this one has no compute-capability floor of 8.0: IMTalker is
plain fp32 convolutions and attention, so Turing (7.5) is fine.


In [ ]:
sh("nvidia-smi --query-gpu=index,name,compute_cap,memory.total,driver_version "
   "--format=csv || echo 'NO GPU VISIBLE'", cwd="/")
sh("echo; df -h /workspace / 2>/dev/null | sed -n '1,5p'; echo; uname -m", cwd="/")

out = subprocess.run(["bash", "-c", "nvidia-smi --query-gpu=compute_cap,memory.total "
                      "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout
if out.strip():
    cap, mem = [x.strip() for x in out.strip().splitlines()[0].split(",")]
    gib = float(mem) / 1024
    print("\ncompute capability %s, %.0f GiB of VRAM" % (cap, gib))
    if float(cap) < 7.0:
        print("!! IMTalker needs compute capability >= 7.0")
    elif gib < 8:
        print("!! under 8 GiB of VRAM -- expect CUDA OOM at 512x512; see section 12")
    else:
        print("ok")

free_gb = shutil.disk_usage("/workspace" if os.path.isdir("/workspace") else "/").free / 2**30
print("%.0f GiB free on the work volume%s" % (free_gb, "" if free_gb >= 20 else "  !! want >= 20 GiB"))


## 3 · System libraries

`ffmpeg` to mux the audio onto the rendered frames, `libsndfile1` for soundfile/librosa,
`libgl1` + `libglib2.0-0` for OpenCV. Already present in most RunPod images, in which case
this is a no-op.


In [ ]:
sh("export DEBIAN_FRONTEND=noninteractive && "
   "apt-get update -qq && "
   "apt-get install -y -qq --no-install-recommends "
   "  git curl ca-certificates bzip2 "
   "  ffmpeg libsndfile1 libgl1 libglib2.0-0 "
   "&& echo '--- installed ---' && ffmpeg -version | head -1", cwd="/")


## 4 · Clone the repo

Only the Python packages under `src/imtalker` are used — `generator/` (the flow-matching
transformer and the Wav2Vec2 subclass) and `renderer/` (the implicit-motion renderer).
Re-running this cell on an existing clone just fetches and fast-forwards it; nothing in
`/workspace/imtalker_only` is touched.


In [ ]:
os.makedirs(os.path.dirname(REPO_DIR), exist_ok=True)
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(REPO_DIR, "already exists - updating")
    # FETCH_HEAD, not origin/HEAD: a --depth 1 clone has no remote HEAD ref
    sh("git -C %s fetch --depth 1 origin %s && git -C %s reset --hard FETCH_HEAD"
       % (REPO_DIR, BRANCH or "HEAD", REPO_DIR), cwd="/", check=False)
else:
    branch = ("--branch " + BRANCH) if BRANCH else ""
    sh("git clone --depth 1 %s %s %s" % (branch, REPO_URL, REPO_DIR), cwd="/")

for need in ("generator/FM.py", "generator/options/base_options.py", "renderer/models.py"):
    p = os.path.join(IMTALKER_DIR, need)
    assert os.path.isfile(p), "the clone is missing %s" % p
print("\nIMTalker package:", IMTALKER_DIR)
sh("ls -la %s | sed -n '1,8p'" % IMTALKER_DIR, cwd="/")


## 5 · Write the inference-only build files

The repo's `setup_env_x86.sh` builds the environment for the **whole** speech-to-speech
server: moshi, sphn, TensorRT, fastapi, uvicorn — ~8 GB of wheels, most of which this
notebook never imports. The three files below are its inference-only counterpart and are
written into `/workspace/imtalker_only`, so the repo's own scripts stay untouched.

| file | what it is for |
|---|---|
| `requirements-imtalker-torch.txt` | torch 2.8.0 / torchvision 0.23.0 / torchaudio 2.8.0, CUDA 12.8 builds (Ampere, Ada, Blackwell `sm_120`) |
| `requirements-imtalker.txt` | transformers 4.30.2, timm, torchdiffeq, librosa, OpenCV, numpy 1.26.4 — and nothing else |
| `setup_env_imtalker.sh` | Miniforge x86_64 + a Python 3.10 env in `./env`, then the two files above |

**Why Python 3.10 and transformers 4.30.2:** `src/imtalker/generator/wav2vec2.py`
subclasses `Wav2Vec2Model` and calls into `.feature_extractor`, `.feature_projection`,
`._mask_hidden_states` and `.encoder` directly. Newer transformers majors rename parts of
that internal surface, and 4.30.2 / tokenizers 0.13.3 publish no wheels for Python 3.12+.


In [ ]:
%%writefile requirements-imtalker-torch.txt
# torch stack for IMTalker inference on any x86_64 NVIDIA GPU -- CUDA 12.8 builds.
# cu128 covers Ampere (A100/A6000), Ada (4090/L40S) and Blackwell (5090/B200, sm_120).
# Same versions as the repo's Jetson and x86 builds, so the checkpoints behave identically.
#     pip install -r requirements-imtalker-torch.txt
--index-url https://download.pytorch.org/whl/cu128
--extra-index-url https://pypi.org/simple
torch==2.8.0+cu128
torchaudio==2.8.0+cu128
torchvision==0.23.0+cu128


In [ ]:
%%writefile requirements-imtalker.txt
# Everything IMTalker inference needs, and nothing the speech-to-speech server needs.
# Install AFTER requirements-imtalker-torch.txt (setup_env_imtalker.sh does both).
#
# The versions are the ones the repo's own requirements.txt pins, so generator.ckpt and
# renderer.ckpt load into exactly the modules they were saved from. The two that matter:
#   transformers 4.30.2 + tokenizers 0.13.3 -- src/imtalker/generator/wav2vec2.py
#       subclasses Wav2Vec2Model and reaches into .feature_extractor, .feature_projection,
#       ._mask_hidden_states and .encoder directly; newer majors rename parts of that.
#   numpy 1.26.4 -- opencv 4.9 and transformers 4.30 are built against the numpy 1.x ABI.
numpy==1.26.4
transformers==4.30.2
tokenizers==0.13.3
huggingface_hub==0.24.7
safetensors==0.4.5
timm==1.0.9
einops==0.7.0
torchdiffeq==0.2.5
librosa==0.11.0
soundfile==0.14.0
numba==0.61.2
llvmlite==0.44.0
opencv-python-headless==4.9.0.80
Pillow==10.4.0
scipy==1.15.3
scikit-image==0.25.2
matplotlib==3.8.4
tqdm==4.67.1


In [ ]:
%%writefile setup_env_imtalker.sh
#!/usr/bin/env bash
# setup_env_imtalker.sh -- Python 3.10 environment for IMTalker inference only.
#
#   ./setup_env_imtalker.sh
#
# This is a trimmed cousin of the repo's setup_env_x86.sh. It installs Miniforge
# (conda-forge, free) into ./.miniforge, creates a Python 3.10 env in ./env, then
# torch/torchaudio/torchvision for CUDA 12.8 followed by requirements-imtalker.txt.
# Nothing from the speech-to-speech server is installed: no moshi, no sphn, no
# TensorRT, no fastapi/uvicorn. Downloads ~4 GB, takes ~5-12 min.
#
# Python 3.10 specifically: transformers 4.30.2 / tokenizers 0.13.3 -- which is what
# IMTalker's Wav2Vec2 subclass expects -- have no wheels for 3.12+.
#
# Everything lands under this directory, so on RunPod it belongs on /workspace and
# survives a pod restart. Safe to re-run: finished steps are skipped.
set -euo pipefail
HERE="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"; cd "$HERE"
MF="$HERE/.miniforge"; ENV="$HERE/env"
MF_URL="${MF_URL:-https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh}"
# keep every cache next to the env so it lands on the persistent volume, not the container disk
export CONDA_PKGS_DIRS="$HERE/.cache/conda-pkgs" PIP_CACHE_DIR="$HERE/.cache/pip" TMPDIR="$HERE/.cache/tmp"
mkdir -p "$CONDA_PKGS_DIRS" "$PIP_CACHE_DIR" "$TMPDIR"
log(){ printf '\n\033[1m==> %s\033[0m\n' "$*"; }
warn(){ printf '\033[33mWARNING: %s\033[0m\n' "$*" >&2; }
die(){ printf '\033[31mERROR: %s\033[0m\n' "$*" >&2; exit 1; }

log "1/7  checks"
[ "$(uname -m)" = x86_64 ] || die "this script is for x86_64 (on a Jetson use the repo's setup_env.sh)"
command -v nvidia-smi >/dev/null || die "nvidia-smi not found -- no NVIDIA driver visible in this container"
nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader | sed 's/^/    /'

log "2/7  Miniforge -> $MF"
if [ ! -x "$MF/bin/conda" ]; then
  curl -fL --retry 3 -o "$TMPDIR/miniforge.sh" "$MF_URL" || die "could not download Miniforge from $MF_URL"
  bash "$TMPDIR/miniforge.sh" -b -p "$MF" > "$TMPDIR/miniforge_install.log" 2>&1 \
    || { tail -20 "$TMPDIR/miniforge_install.log"; die "Miniforge install failed"; }
  rm -f "$TMPDIR/miniforge.sh"
fi
"$MF/bin/conda" --version | sed 's/^/    /'

log "3/7  Python 3.10 env -> $ENV"
if [ ! -x "$ENV/bin/python" ]; then
  "$MF/bin/conda" create -y -q -p "$ENV" python=3.10 pip > "$TMPDIR/conda_create.log" 2>&1 \
    || { tail -20 "$TMPDIR/conda_create.log"; die "conda create failed"; }
fi
PY="$ENV/bin/python"
"$PY" -c 'import sys; assert sys.version_info[:2] == (3, 10), sys.version' || die "env python is not 3.10"
"$PY" -m pip install -q --upgrade pip

log "4/7  torch / torchvision / torchaudio for CUDA 12.8 (~3 GB)"
if ! "$PY" -c 'import torch, sys; sys.exit(0 if torch.version.cuda and torch.cuda.is_available() else 1)' 2>/dev/null; then
  "$PY" -m pip install -r "$HERE/requirements-imtalker-torch.txt" \
    || die "torch install failed (index: https://download.pytorch.org/whl/cu128)"
fi
"$PY" -c 'import torch, sys; print("    torch", torch.__version__, "CUDA", torch.version.cuda, "|", torch.cuda.get_device_name(0)); sys.exit(0 if torch.cuda.is_available() else 1)' \
  || die "torch cannot see the GPU"

log "5/7  IMTalker inference requirements"
"$PY" -m pip install -r "$HERE/requirements-imtalker.txt" || die "pip install -r requirements-imtalker.txt failed"

log "6/7  face-alignment (optional: only the face detector behind --crop)"
# --no-deps is deliberate. face-alignment declares plain `numpy` and `opencv-python` with
# no upper bound, so a normal install upgrades numpy to 2.x -- which the opencv 4.9 wheel
# cannot load ("numpy 2.x cannot be run in ... _ARRAY_API not found") -- and drops the GUI
# opencv on top of the headless one. Everything face-alignment actually imports (torch,
# numpy, scipy, scikit-image, tqdm, numba, cv2) is already installed by the step above.
# A failure here is not fatal: imtalker_infer.py falls back to a centre crop, and a
# 512x512 portrait that is already framed on the face needs no crop at all.
"$PY" -m pip install --no-deps "face-alignment==1.4.1" >/dev/null 2>&1 \
  && echo "    face-alignment 1.4.1 (optional --crop support): ok" \
  || warn "face-alignment did not install -- run without --crop, or pre-crop the image yourself"

log "7/7  pin repair (undo anything an earlier run pulled out from under the pins)"
# The GUI opencv overwrites the headless build's cv2, so if it is present replace it.
if "$PY" -m pip show opencv-python >/dev/null 2>&1 || "$PY" -m pip show opencv-contrib-python >/dev/null 2>&1; then
  echo "    removing the GUI opencv and restoring opencv-python-headless"
  "$PY" -m pip uninstall -y -q opencv-python opencv-contrib-python >/dev/null 2>&1 || true
  "$PY" -m pip install -q --force-reinstall --no-deps "opencv-python-headless==4.9.0.80" \
    || die "could not reinstall opencv-python-headless"
fi
# numpy must stay on the 1.x ABI: opencv 4.9 and transformers 4.30 are compiled against it.
if ! "$PY" -c 'import numpy, sys; sys.exit(0 if numpy.__version__ == "1.26.4" else 1)' 2>/dev/null; then
  echo "    restoring numpy==1.26.4"
  "$PY" -m pip install -q --force-reinstall --no-deps "numpy==1.26.4" || die "could not pin numpy to 1.26.4"
fi
echo "    pins ok"

"$PY" - <<'T' || die "installed, but the environment check failed -- re-run this cell once; it repairs the pins"
import sys
import torch, torchvision, transformers, timm, torchdiffeq, librosa, cv2, numpy, PIL
assert torch.cuda.is_available(), "torch has no CUDA"
assert numpy.__version__.startswith("1."), (
    f"numpy {numpy.__version__} is on the 2.x ABI, which the opencv 4.9 and transformers "
    "4.30 wheels cannot load. Re-run this cell: step 7/7 pins it back to 1.26.4."
)
assert transformers.__version__ == "4.30.2", f"transformers {transformers.__version__} (want 4.30.2)"
try:
    import face_alignment
    fa = f" | face_alignment {face_alignment.__version__}"
except Exception:
    fa = " | face_alignment MISSING (--crop unavailable)"
cc = torch.cuda.get_device_capability(0)
print(f"    python {sys.version.split()[0]} | torch {torch.__version__} (CUDA {torch.version.cuda}) | "
      f"transformers {transformers.__version__} | numpy {numpy.__version__} | "
      f"opencv {cv2.__version__}{fa}")
print(f"    {torch.cuda.get_device_name(0)} sm_{cc[0]}{cc[1]} "
      f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GiB")
T
rm -rf "$TMPDIR"
log "ENV READY: $ENV"


## 6 · Build the Python 3.10 environment

~5–12 minutes and ~4 GB the first time. Safe to re-run: Miniforge, the env and every
finished pip step are skipped, so this is also the cell to re-run after a pod restart to
confirm the env on the volume still works.

Two things the script does that are worth knowing about, because getting them wrong breaks
the env in a way that only shows up much later:

* **`face-alignment` is installed with `--no-deps`.** It declares plain `numpy` and
  `opencv-python` with no upper bound, so a normal install upgrades numpy to 2.x — which
  the opencv 4.9 and transformers 4.30 wheels cannot load (`_ARRAY_API not found`) — and
  drops the GUI OpenCV on top of the headless one. Everything it actually imports is
  already installed by the step before it. It is also still allowed to fail: it is only
  the face detector that picks the crop window for `CROP = True`.
* **step 7/7 is a pin repair.** It puts numpy back on 1.26.4 and restores
  `opencv-python-headless` if anything moved them, so re-running this cell fixes an env
  that a stray `pip install` has broken rather than making you rebuild it.


In [ ]:
sh("chmod +x setup_env_imtalker.sh && ./setup_env_imtalker.sh")
assert os.path.isfile(PY_ENV), "the env was not created at %s" % PY_ENV
print("\nenv python:", PY_ENV)


## 7 · Download the IMTalker checkpoints

Three things, ~3 GB in total, from the public and ungated
[`cbsjtu01/IMTalker`](https://huggingface.co/cbsjtu01/IMTalker) — no Hugging Face token:

| file | what it is |
|---|---|
| `generator.ckpt` | the flow-matching motion generator: Wav2Vec2 features → 32-d motion latents at 25 fps |
| `renderer.ckpt` | the implicit-motion renderer: reference image + motion latent → a 512×512 frame |
| `wav2vec2-base-960h/` | the frozen audio encoder the generator was trained against; it must be local, `FM.py` loads it with `local_files_only=True` |

Already-downloaded files are skipped. If you set `GENERATOR_OVERRIDE` /
`RENDERER_OVERRIDE` in section 1 (a fine-tune of your own, e.g. a `generator_last.ckpt`),
those are used instead and nothing is fetched for them — `imtalker_infer.py` reports how
many tensors actually matched the model, so a mismatched checkpoint is loud rather than
silently producing a still face.


In [ ]:
os.makedirs(CKPT_DIR, exist_ok=True)

WAV2VEC_FILES = [
    "wav2vec2-base-960h/config.json",
    "wav2vec2-base-960h/preprocessor_config.json",
    "wav2vec2-base-960h/feature_extractor_config.json",
    "wav2vec2-base-960h/pytorch_model.bin",
]
wanted = list(WAV2VEC_FILES)
if not GENERATOR_OVERRIDE:
    wanted.append("generator.ckpt")
if not RENDERER_OVERRIDE:
    wanted.append("renderer.ckpt")

# Run the download inside the env we just built, so its huggingface_hub is the one used.
dl = textwrap.dedent("""
    import os, sys
    from huggingface_hub import hf_hub_download
    repo, dest, files = sys.argv[1], sys.argv[2], sys.argv[3:]
    for name in files:
        out = os.path.join(dest, name)
        if os.path.exists(out) and os.path.getsize(out) > 1024:
            print("    have   %s (%.1f MB)" % (name, os.path.getsize(out) / 1e6))
            continue
        print("    get    %s ..." % name, flush=True)
        hf_hub_download(repo_id=repo, filename=name, repo_type="model",
                        local_dir=dest, local_dir_use_symlinks=False)
        print("    stored %s (%.1f MB)" % (name, os.path.getsize(out) / 1e6), flush=True)
""")
Path(os.path.join(WORK, "_fetch_ckpts.py")).write_text(dl, encoding="utf-8")
sh('"%s" _fetch_ckpts.py %s %s %s' % (PY_ENV, HF_CKPT_REPO, CKPT_DIR, " ".join(wanted)))

GENERATOR_PATH = GENERATOR_OVERRIDE or os.path.join(CKPT_DIR, "generator.ckpt")
RENDERER_PATH  = RENDERER_OVERRIDE  or os.path.join(CKPT_DIR, "renderer.ckpt")
WAV2VEC_PATH   = os.path.join(CKPT_DIR, "wav2vec2-base-960h")

for label, p in (("generator", GENERATOR_PATH), ("renderer", RENDERER_PATH)):
    assert os.path.isfile(p), "%s checkpoint missing: %s" % (label, p)
    print("%-10s %s  (%.0f MB)" % (label, p, os.path.getsize(p) / 1e6))
assert os.path.isfile(os.path.join(WAV2VEC_PATH, "config.json")), WAV2VEC_PATH
print("%-10s %s" % ("wav2vec2", WAV2VEC_PATH))


## 8 · The inference script

`imtalker_infer.py` is the same model path as the repo's `src/imtalker/app.py` and
`src/imtalker/generator/generate.py` — build `IMTRenderer` and `FMGenerator`, load the two
checkpoints, encode the reference image, sample motion from the audio, decode frame by
frame — with four changes that matter for a batch job in a notebook:

* **every model hyper-parameter comes from the repo's own `options/base_options.py`**, so
  the script cannot drift away from what the checkpoints were saved with. Only
  `input_size`, `fps`, the sampler knobs and the paths are overridden.
* **frames are streamed to disk** as they are decoded instead of being stacked into one
  tensor. `generate.py` holds all `T` frames at once: a minute of audio is 1500 frames,
  which is ~4.7 GB of fp32 at 512×512 and a reliable OOM.
* **the output range is detected, not assumed.** `generate.py` does
  `clamp(-1, 1) * 255 → ByteTensor`, which wraps every negative pixel around to white if
  the renderer head is tanh-like. This script reads the first frame's range, picks
  `[-1,1]` or `[0,1]` once, and keeps that choice for the whole clip.
* **H.264 + yuv420p + faststart**, so the result plays in a browser and in the notebook.
  `generate.py` emits `mp4v`, which Chrome will not decode.

It takes a `--imtalker_dir` and puts both that directory (for `generator.FM` /
`renderer.models`) and its `generator/` subdirectory (for `options.base_options`, which
`generate.py` imports as a top-level module) on `sys.path`.


In [ ]:
%%writefile imtalker_infer.py
#!/usr/bin/env python3
"""IMTalker-only offline inference: one image + one audio file -> talking-head MP4.

Nothing from the speech-to-speech stack is touched here. The only things loaded are
IMTalker's flow-matching motion generator (`generator.ckpt`), IMTalker's renderer
(`renderer.ckpt`) and the frozen Wav2Vec2 audio encoder the generator was trained
against -- i.e. exactly the `app.py` / `generator/generate.py` path of the repo,
driven from the command line and streamed straight to an mp4.

    python imtalker_infer.py \
        --imtalker_dir /workspace/speech2face_imtalker/src/imtalker \
        --image  .../rb.jpeg \
        --audio  .../rb_wav.wav \
        --out    .../rb_imtalker.mp4 \
        --generator_path .../generator.ckpt \
        --renderer_path  .../renderer.ckpt \
        --wav2vec_model_path .../wav2vec2-base-960h
"""
from __future__ import annotations

import argparse
import json
import shutil
import subprocess
import sys
import time
from pathlib import Path

import cv2
import librosa
import numpy as np
import torch
import torchvision.transforms as transforms
from PIL import Image
from transformers import Wav2Vec2FeatureExtractor


# --------------------------------------------------------------------------- paths
def add_imtalker_to_path(imtalker_dir: Path) -> None:
    """`generator.FM` / `renderer.models` need src/imtalker on sys.path, and
    `options.base_options` (imported from inside generator/) needs src/imtalker/generator.
    """
    for p in (imtalker_dir / "generator", imtalker_dir):
        s = str(p.resolve())
        if s not in sys.path:
            sys.path.insert(0, s)


# --------------------------------------------------------------------------- options
def build_opt(args: argparse.Namespace):
    """Take every model hyper-parameter from the repo's own BaseOptions defaults, so this
    script cannot drift away from the checkpoints, and override only what inference needs.
    """
    from options.base_options import BaseOptions

    argv = [
        "imtalker_infer.py",
        "--input_size", str(args.input_size),
        "--fps", str(args.fps),
        "--sampling_rate", "16000",
        "--wav2vec_sec", str(args.wav2vec_sec),
        "--wav2vec_model_path", str(args.wav2vec_model_path),
        "--num_prev_frames", str(args.num_prev_frames),
        "--nfe", str(args.nfe),
        "--a_cfg_scale", str(args.a_cfg_scale),
        "--seed", str(args.seed),
        "--torchdiffeq_ode_method", args.ode_method,
    ]
    if args.fix_noise_seed:
        argv.append("--fix_noise_seed")

    old_argv, sys.argv = sys.argv, argv
    try:
        opt = BaseOptions().parse()
    finally:
        sys.argv = old_argv

    # FMGenerator._device() reads opt.rank; the Gradio app config also carries opt.device.
    opt.rank = args.device
    opt.device = args.device
    opt.ngpus = 1
    opt.generator_path = str(args.generator_path)
    opt.renderer_path = str(args.renderer_path)
    # the live server's eye-blink compositing is not part of this offline path
    opt.enable_eye_blink_composite = False
    return opt


# --------------------------------------------------------------------------- inputs
def crop_to_face(img: Image.Image) -> Image.Image:
    """Square crop around the largest detected face with 0.8*bbox of padding on each
    side -- the framing app.py uses. Degrades to a centre crop when no face is found or
    when face_alignment is not installed.
    """
    arr = np.array(img)
    h, w = arr.shape[:2]
    boxes = []
    try:
        import face_alignment  # heavy + optional: only imported when --crop is asked for

        fa = face_alignment.FaceAlignment(
            face_alignment.LandmarksType.TWO_D, device="cpu", flip_input=False
        )
        detected = fa.face_detector.detect_from_image(arr)
        boxes = [b for b in (detected or []) if float(b[4]) > 0.5]
    except Exception as exc:  # noqa: BLE001 - any failure degrades to a centre crop
        print(f"[crop] face detection unavailable ({exc}) -> centre crop", flush=True)

    if not boxes:
        print("[crop] no usable face box -> centre crop", flush=True)
        half = min(w, h) // 2
        cx, cy = w // 2, h // 2
        x1, y1, x2, y2 = cx - half, cy - half, cx + half, cy + half
    else:
        bx1, by1, bx2, by2 = (float(v) for v in boxes[0][:4])
        cx, cy = (bx1 + bx2) / 2.0, (by1 + by2) / 2.0
        half = int(max(bx2 - bx1, by2 - by1) * 0.8)
        x1, y1, x2, y2 = int(cx - half), int(cy - half), int(cx + half), int(cy + half)
        # slide the window back inside the image rather than letting it shrink off-centre
        if x1 < 0:
            x2 -= x1
            x1 = 0
        if y1 < 0:
            y2 -= y1
            y1 = 0
        if x2 > w:
            x1 -= x2 - w
            x2 = w
        if y2 > h:
            y1 -= y2 - h
            y2 = h
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        side = min(x2 - x1, y2 - y1)
        x2, y2 = x1 + side, y1 + side

    print(f"[crop] {w}x{h} -> box ({x1},{y1})-({x2},{y2})", flush=True)
    return Image.fromarray(arr[y1:y2, x1:x2])


def load_reference(path: Path, input_size: int, crop: bool) -> Image.Image:
    img = Image.open(path).convert("RGB")
    print(f"[input] image {path} {img.size[0]}x{img.size[1]}", flush=True)
    if crop:
        img = crop_to_face(img)
    if img.size != (input_size, input_size):
        img = img.resize((input_size, input_size), Image.LANCZOS)
    return img


def load_audio(path: Path, sampling_rate: int) -> tuple[np.ndarray, float]:
    speech, sr = librosa.load(str(path), sr=sampling_rate, mono=True)
    speech = np.asarray(speech, dtype=np.float32)
    seconds = float(len(speech)) / float(sr)
    print(f"[input] audio {path} {seconds:.2f}s -> {sr} Hz, {len(speech)} samples", flush=True)
    if len(speech) == 0:
        raise ValueError(f"{path} decoded to zero samples")
    return speech, seconds


# --------------------------------------------------------------------------- weights
def load_renderer_ckpt(model, path: str, prefix: str = "gen.") -> None:
    ckpt = torch.load(path, map_location="cpu")
    state = ckpt.get("state_dict", ckpt) if isinstance(ckpt, dict) else ckpt
    cleaned = {k[len(prefix):]: v for k, v in state.items() if k.startswith(prefix)}
    if not cleaned:  # a checkpoint saved without the Lightning "gen." prefix
        cleaned = dict(state)
    missing, unexpected = model.load_state_dict(cleaned, strict=False)
    matched = len(cleaned) - len(unexpected)
    print(
        f"[weights] renderer {path}: matched {matched}/{len(cleaned)} tensors "
        f"(missing {len(missing)}, unexpected {len(unexpected)})",
        flush=True,
    )
    if matched <= 0:
        raise RuntimeError(f"no renderer weight matched the model: {path}")


def load_generator_ckpt(model, path: str, prefix: str = "model.") -> None:
    ckpt = torch.load(path, map_location="cpu")
    state = ckpt.get("state_dict", ckpt) if isinstance(ckpt, dict) else ckpt
    if isinstance(state.get("model"), dict):
        state = state["model"]
    cleaned = {k[len(prefix):]: v for k, v in state.items() if k.startswith(prefix)}
    if not cleaned:
        cleaned = dict(state)
    copied = 0
    with torch.no_grad():
        for name, param in model.named_parameters():
            src = cleaned.get(name)
            if src is not None and tuple(src.shape) == tuple(param.shape):
                param.copy_(src.to(param.device, dtype=param.dtype))
                copied += 1
        for name, buf in model.named_buffers():
            src = cleaned.get(name)
            if src is not None and tuple(src.shape) == tuple(buf.shape):
                buf.copy_(src.to(buf.device, dtype=buf.dtype))
    total = sum(1 for _ in model.named_parameters())
    print(f"[weights] generator {path}: copied {copied}/{total} parameter tensors", flush=True)
    if copied == 0:
        raise RuntimeError(f"no generator weight matched the model: {path}")


# --------------------------------------------------------------------------- video out
class Mp4Writer:
    """cv2.VideoWriter opened lazily on the first frame, so the frame size comes from the
    renderer instead of an assumption, and decoded frames never pile up in VRAM.
    """

    def __init__(self, path: Path, fps: float):
        self.path = path
        self.fps = float(fps)
        self._w = None
        self.frames = 0

    def write(self, rgb: np.ndarray) -> None:
        if self._w is None:
            h, w = rgb.shape[:2]
            self._w = cv2.VideoWriter(
                str(self.path), cv2.VideoWriter_fourcc(*"mp4v"), self.fps, (w, h)
            )
            if not self._w.isOpened():
                raise RuntimeError(f"cv2 could not open a video writer for {self.path}")
            print(f"[video] writing {w}x{h} @ {self.fps:g} fps -> {self.path}", flush=True)
        self._w.write(cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
        self.frames += 1

    def close(self) -> None:
        if self._w is not None:
            self._w.release()
            self._w = None


def mux(silent: Path, audio: Path, out: Path, crf: int) -> bool:
    """Re-encode the mp4v intermediate to H.264 and attach the original audio."""
    ffmpeg = shutil.which("ffmpeg")
    if not ffmpeg:
        print("[mux] ffmpeg not on PATH -- keeping the silent mp4v file", flush=True)
        return False
    cmd = [
        ffmpeg, "-y", "-loglevel", "error",
        "-i", str(silent),
        "-i", str(audio),
        "-map", "0:v:0", "-map", "1:a:0",
        "-c:v", "libx264", "-crf", str(crf), "-preset", "medium", "-pix_fmt", "yuv420p",
        "-c:a", "aac", "-b:a", "192k", "-ac", "1",
        "-movflags", "+faststart",
        "-shortest",
        str(out),
    ]
    rc = subprocess.run(cmd).returncode
    if rc != 0:
        print(f"[mux] ffmpeg failed (exit {rc}) -- keeping the silent mp4v file", flush=True)
        return False
    return True


# --------------------------------------------------------------------------- main
def parse_args() -> argparse.Namespace:
    ap = argparse.ArgumentParser(description="IMTalker image + audio -> talking-head video")
    ap.add_argument("--imtalker_dir", required=True, help="path to <clone>/src/imtalker")
    ap.add_argument("--image", required=True)
    ap.add_argument("--audio", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--generator_path", required=True)
    ap.add_argument("--renderer_path", required=True)
    ap.add_argument("--wav2vec_model_path", required=True)
    ap.add_argument("--device", default="cuda")
    ap.add_argument("--input_size", type=int, default=512)
    ap.add_argument("--fps", type=float, default=25.0)
    ap.add_argument("--wav2vec_sec", type=float, default=2.0,
                    help="generator clip length in seconds; must match the checkpoint (2.0)")
    ap.add_argument("--num_prev_frames", type=int, default=10)
    ap.add_argument("--nfe", type=int, default=10, help="ODE steps of the flow-matching sampler")
    ap.add_argument("--a_cfg_scale", type=float, default=2.0, help="audio classifier-free guidance")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--fix_noise_seed", action="store_true")
    ap.add_argument("--ode_method", default="euler")
    ap.add_argument("--crop", action="store_true",
                    help="auto-crop around the detected face first (needs face_alignment)")
    ap.add_argument("--crf", type=int, default=18)
    ap.add_argument("--keep_silent", action="store_true", help="keep the intermediate silent mp4")
    ap.add_argument("--tf32", action="store_true", help="allow TF32 matmuls (faster, slightly less exact)")
    return ap.parse_args()


def main() -> None:
    args = parse_args()
    imtalker_dir = Path(args.imtalker_dir).resolve()
    if not (imtalker_dir / "generator" / "FM.py").is_file():
        raise SystemExit(f"--imtalker_dir does not look like src/imtalker: {imtalker_dir}")
    for p in (args.image, args.audio, args.generator_path, args.renderer_path):
        if not Path(p).exists():
            raise SystemExit(f"missing input: {p}")
    if not (Path(args.wav2vec_model_path) / "config.json").is_file():
        raise SystemExit(f"--wav2vec_model_path has no config.json: {args.wav2vec_model_path}")

    add_imtalker_to_path(imtalker_dir)
    from generator.FM import FMGenerator  # noqa: E402  (needs the sys.path above)
    from renderer.models import IMTRenderer  # noqa: E402

    if args.device.startswith("cuda") and not torch.cuda.is_available():
        raise SystemExit("--device cuda but torch.cuda.is_available() is False")
    if args.tf32:
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    opt = build_opt(args)
    device = torch.device(args.device)

    out = Path(args.out).resolve()
    out.parent.mkdir(parents=True, exist_ok=True)
    silent = out.with_name(out.stem + "_silent.mp4")

    t0 = time.time()
    print("[load] building the renderer and the motion generator ...", flush=True)
    renderer = IMTRenderer(opt)
    generator = FMGenerator(opt)
    load_renderer_ckpt(renderer, opt.renderer_path)
    load_generator_ckpt(generator, opt.generator_path)
    renderer = renderer.to(device).eval()
    generator = generator.to(device).eval()
    print(f"[load] ready in {time.time() - t0:.1f}s", flush=True)

    img = load_reference(Path(args.image), opt.input_size, args.crop)
    speech, audio_seconds = load_audio(Path(args.audio), opt.sampling_rate)

    extractor = Wav2Vec2FeatureExtractor.from_pretrained(
        args.wav2vec_model_path, local_files_only=True
    )
    a = extractor(
        speech, sampling_rate=opt.sampling_rate, return_tensors="pt"
    ).input_values[0].unsqueeze(0).to(device)

    transform = transforms.Compose([
        transforms.Resize((opt.input_size, opt.input_size)),
        transforms.ToTensor(),
    ])
    s = transform(img).unsqueeze(0).to(device)

    writer = Mp4Writer(silent, opt.fps)
    t_sample = t_render = 0.0
    with torch.no_grad():
        f_r, g_r = renderer.dense_feature_encoder(s)
        t_r = renderer.latent_token_encoder(s)
        if isinstance(t_r, tuple):
            t_r = t_r[0]

        print(
            f"[sample] flow matching: nfe={args.nfe} a_cfg_scale={args.a_cfg_scale} "
            f"seed={args.seed}",
            flush=True,
        )
        t1 = time.time()
        torch.manual_seed(args.seed)
        motion = generator.sample(
            {"s": s, "a": a, "ref_x": t_r, "pose": None, "cam": None, "gaze": None},
            a_cfg_scale=float(args.a_cfg_scale),
            nfe=int(args.nfe),
            seed=int(args.seed),
        )
        t_sample = time.time() - t1
        total = int(motion.shape[1])
        print(
            f"[sample] {total} motion frames in {t_sample:.1f}s "
            f"({total / max(float(opt.fps), 1e-6):.2f}s of video)",
            flush=True,
        )

        ta_r = renderer.adapt(t_r, g_r)
        m_r = renderer.latent_token_decoder(ta_r)

        t1 = time.time()
        signed = None
        for t in range(total):
            ta_c = renderer.adapt(motion[:, t, ...], g_r)
            m_c = renderer.latent_token_decoder(ta_c)
            frame = renderer.decode(m_c, m_r, f_r)
            arr = frame[0].detach().float().permute(1, 2, 0).cpu().numpy()
            if signed is None:
                # some renderer heads are tanh-like ([-1,1]) and some sigmoid-like ([0,1]);
                # decide once from the first frame and keep that choice for the whole clip
                signed = bool(arr.min() < -0.01)
                print(
                    f"[render] first frame range [{arr.min():.3f}, {arr.max():.3f}] -> "
                    f"treating output as {'[-1,1]' if signed else '[0,1]'}",
                    flush=True,
                )
            if signed:
                arr = (arr + 1.0) / 2.0
            writer.write((np.clip(arr, 0.0, 1.0) * 255.0).astype(np.uint8))
            if (t + 1) % 25 == 0 or t + 1 == total:
                done = t + 1
                print(
                    f"[render] {done}/{total} frames "
                    f"({done / max(time.time() - t1, 1e-6):.1f} fps)",
                    flush=True,
                )
        t_render = time.time() - t1
    writer.close()
    if writer.frames == 0:
        raise SystemExit("the renderer produced no frames")

    has_audio = mux(silent, Path(args.audio), out, args.crf)
    if not has_audio:
        shutil.move(str(silent), str(out))
    elif not args.keep_silent:
        silent.unlink(missing_ok=True)

    meta = {
        "output": str(out),
        "image": str(Path(args.image).resolve()),
        "audio": str(Path(args.audio).resolve()),
        "generator_path": opt.generator_path,
        "renderer_path": opt.renderer_path,
        "frames": writer.frames,
        "fps": float(opt.fps),
        "video_seconds": round(writer.frames / float(opt.fps), 3),
        "audio_seconds": round(audio_seconds, 3),
        "has_audio_track": has_audio,
        "nfe": int(args.nfe),
        "a_cfg_scale": float(args.a_cfg_scale),
        "seed": int(args.seed),
        "crop": bool(args.crop),
        "sample_seconds": round(t_sample, 2),
        "render_seconds": round(t_render, 2),
        "total_seconds": round(time.time() - t0, 2),
        "size_bytes": out.stat().st_size,
    }
    out.with_suffix(".json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
    print(json.dumps(meta, indent=2), flush=True)


if __name__ == "__main__":
    main()


## 9 · Check the inputs

A quick look before spending GPU time: the reference frame as the model will actually see
it (square, 512×512) and what `ffprobe` makes of the audio. `rb_wav.wav` is 48 kHz mono,
~7.8 s; `librosa` resamples it to the 16 kHz the Wav2Vec2 encoder expects, and the video
comes out at `ceil(samples × 25 / 16000)` frames — about 196 here.


In [ ]:
from IPython.display import display, Image as IPyImage

IMAGE_PATH = resolve(IMAGE, REPO_DIR)
AUDIO_PATH = resolve(AUDIO, REPO_DIR)
OUT_PATH   = resolve(OUT, WORK)
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

for label, p in (("image", IMAGE_PATH), ("audio", AUDIO_PATH)):
    assert os.path.isfile(p), ("%s not found: %s\n"
        "Set IMAGE / AUDIO in section 1, or upload the file to /workspace." % (label, p))
    print("%-6s %s  (%.2f MB)" % (label, p, os.path.getsize(p) / 1e6))

sh('ffprobe -hide_banner -v error -show_entries '
   'stream=codec_name,sample_rate,channels,duration -of default=noprint_wrappers=1 "%s"'
   % AUDIO_PATH, check=False)

# the reference frame exactly as the model receives it
prep = textwrap.dedent("""
    import sys
    from pathlib import Path
    work, src, dst, size, crop = sys.argv[1:6]
    sys.path.insert(0, work)
    from imtalker_infer import load_reference
    load_reference(Path(src), int(size), crop == "1").save(dst)
    print("preview ->", dst)
""")
Path(os.path.join(WORK, "_preview_ref.py")).write_text(prep, encoding="utf-8")
preview = os.path.join(WORK, "outputs", "reference_preview.png")
sh('"%s" _preview_ref.py "%s" "%s" "%s" 512 %d'
   % (PY_ENV, WORK, IMAGE_PATH, preview, 1 if CROP else 0))
display(IPyImage(filename=preview, width=320))


## 10 · Generate the video

Everything up to here was setup; this is the actual run. On an A100 or a 4090 expect a few
seconds for the sampler plus roughly 15–30 rendered frames per second, so `rb_wav.wav`
(~196 frames) finishes in well under a minute. The first run also pays a few seconds for
cuDNN autotuning.

The script prints the number of tensors each checkpoint matched, the motion-frame count,
the detected output range and a progress line every 25 frames, then a JSON summary that is
also written next to the video as `<name>.json`.


In [ ]:
cmd = (
    '"%s" imtalker_infer.py'
    ' --imtalker_dir "%s"'
    ' --image "%s" --audio "%s" --out "%s"'
    ' --generator_path "%s" --renderer_path "%s" --wav2vec_model_path "%s"'
    ' --device cuda --input_size 512 --fps %g'
    ' --nfe %d --a_cfg_scale %g --seed %d --crf %d --tf32'
    % (PY_ENV, IMTALKER_DIR, IMAGE_PATH, AUDIO_PATH, OUT_PATH,
       GENERATOR_PATH, RENDERER_PATH, WAV2VEC_PATH,
       FPS, NFE, A_CFG_SCALE, SEED, CRF)
)
if CROP:
    cmd += " --crop"

t0 = time.time()
sh(cmd)
print("\nwall clock: %.1fs" % (time.time() - t0))

meta = json.loads(Path(OUT_PATH).with_suffix(".json").read_text())
print("\nvideo : %s  (%.1f MB)" % (meta["output"], meta["size_bytes"] / 1e6))
print("frames: %d at %g fps = %.2fs video / %.2fs audio"
      % (meta["frames"], meta["fps"], meta["video_seconds"], meta["audio_seconds"]))
if not meta["has_audio_track"]:
    print("!! ffmpeg could not mux the audio -- the file is video-only")


## 11 · Play it, and get it off the pod

The player below embeds the MP4 in the notebook as base64. That is fine for a few seconds
of 512×512 but gets unwieldy past ~30 MB, so for anything long use the printed path with
`runpodctl send`, the pod's file browser, or a JupyterLab download.


In [ ]:
from IPython.display import HTML, display
import base64

size_mb = os.path.getsize(OUT_PATH) / 1e6
if size_mb <= 40:
    b64 = base64.b64encode(Path(OUT_PATH).read_bytes()).decode()
    display(HTML('<video width="512" controls autoplay loop '
                 'src="data:video/mp4;base64,%s"></video>' % b64))
else:
    print("%.0f MB is too big to embed -- download it instead" % size_mb)

print("\nfile        :", OUT_PATH)
print("jupyter     : open the file-browser at /workspace and download it from there")
print("runpodctl   : runpodctl send %s" % OUT_PATH)
print("\nsummary:")
print(json.dumps(meta, indent=2))


## 12 · Knobs, batching and troubleshooting

### Sampler knobs

| knob | default | what moving it does |
|---|---|---|
| `NFE` | `10` | ODE steps of the flow-matching sampler. Below ~5 the mouth gets jittery; above ~25 nothing visibly improves and you just pay for it. The live server runs at 3 because it has a 40 ms budget per chunk — an offline run has no reason to. |
| `A_CFG_SCALE` | `2.0` | audio classifier-free guidance. `1.0` = no guidance, closed and under-articulated mouth. `2.0`–`3.0` = crisper consonants. Past ~3.5 the jaw overshoots and the face starts to wobble. |
| `SEED` | `42` | the initial noise. Same seed + same audio + same image = the same video, bit for bit. Change it if you dislike the head motion you got. |
| `CROP` | `False` | `True` runs the face detector and takes a square crop with 0.8·bbox of padding, which is the framing the model was trained on. Leave it off for an already-square portrait like `rb.jpeg` — a second crop only throws away pixels. |
| `FPS` | `25.0` | **leave this alone.** The generator emits motion at 25 fps by construction; another value stretches the video against the audio. |

### Other inputs

```python
IMAGE = "/workspace/my_photo.jpg"      # any readable path
AUDIO = "/workspace/my_speech.mp3"      # librosa reads mp3/m4a/flac too, via ffmpeg
OUT   = "outputs/my_avatar.mp4"
CROP  = True                            # a wider snapshot needs the crop
```

Re-run section 9 to see the crop, then section 10.

### A whole folder of audio against one image

```python
for wav in sorted(Path("/workspace/clips").glob("*.wav")):
    sh('"%s" imtalker_infer.py --imtalker_dir "%s" --image "%s" --audio "%s" --out "%s"'
       ' --generator_path "%s" --renderer_path "%s" --wav2vec_model_path "%s"'
       ' --device cuda --nfe %d --a_cfg_scale %g --seed %d --tf32'
       % (PY_ENV, IMTALKER_DIR, IMAGE_PATH, wav,
          os.path.join(WORK, "outputs", wav.stem + ".mp4"),
          GENERATOR_PATH, RENDERER_PATH, WAV2VEC_PATH, NFE, A_CFG_SCALE, SEED))
```

Each call reloads the checkpoints (~10 s). For many clips, import `imtalker_infer` and
reuse one loaded pair instead.

### Your own fine-tuned checkpoints

Set `GENERATOR_OVERRIDE` / `RENDERER_OVERRIDE` in section 1 and re-run from section 7.
The loader accepts a raw `state_dict`, a Lightning `{"state_dict": ...}`, a nested
`{"state_dict": {"model": ...}}`, and the `model.` / `gen.` prefixes the repo's training
scripts write. It skips tensors whose shape does not match and prints the count, so check
that line: `copied 3/412` means the checkpoint is not for this architecture, and you would
otherwise get a video of a perfectly still face.

The one checkpoint-coupled hyper-parameter is `--wav2vec_sec 2.0`, the generator's clip
length in frames (`wav2vec_sec × fps = 50`). The live server passes `0.96` or `2.0`
depending on which fine-tune it loads; for `cbsjtu01/IMTalker` it is `2.0`.

### Troubleshooting

| symptom | what to do |
|---|---|
| `numpy 2.x ... _ARRAY_API not found`, or `AssertionError: numpy 2.2.6` in section 6 | something installed a package that moved numpy off 1.26.4. Just re-run section 6: step 7/7 pins numpy back and restores `opencv-python-headless`. If you install anything else into this env yourself, pass `--no-deps` or re-run section 6 afterwards |
| `CUDA out of memory` | nothing accumulates on the GPU here, so this is the model itself. Close any other process on the card (`nvidia-smi`); below 8 GiB, run `--device cpu` — correct, but minutes per frame |
| `copied 0/412 parameter tensors` | the checkpoint is not an IMTalker generator, or the download truncated. Delete it from `checkpoints/` and re-run section 7 |
| video is a still face | the generator matched few tensors (see the line above), or the audio decoded to silence — check `ffprobe` in section 9 |
| `No face detected` / a bad crop | set `CROP = False` and pre-crop the image yourself, or check `outputs/reference_preview.png` from section 9 |
| the mouth is out of sync | `FPS` was changed away from 25, or the audio was trimmed after the video was made. Keep 25 and let `--shortest` do the trimming |
| `local_files_only=True` error on wav2vec | `checkpoints/wav2vec2-base-960h/` is incomplete; delete the directory and re-run section 7 |
| `ffmpeg not on PATH` | re-run section 3. Without it you still get the video, just silent |
| pod restarted, nothing works | re-run sections 1 → 6. Everything is on `/workspace`, so 6 and 7 are no-ops and you are back in ~1 min |
